In [ ]:
import pandas as pd
import ccxt

In [2]:
df_23 = pd.read_csv("BTCUSDT_5m_2023.csv")
df_24 = pd.read_csv("BTCUSDT_5m_2024.csv")
df_25 = pd.read_csv("BTCUSDT_5m_2025.csv")

In [8]:
df = pd.concat([df_23, df_24, df_25], ignore_index=True)

In [9]:
print(df.duplicated().sum())

2


In [10]:
df = df.drop_duplicates()
print(df.duplicated().sum())

0


In [11]:
df.set_index("Open Time", inplace=True)

In [13]:
df = df.drop("Ignore", axis=1)

In [19]:
# Verify Time Continuity

df.index = pd.to_datetime(df.index)
time_diff = df.index.to_series().diff()

print(time_diff.value_counts().head())

Open Time
0 days 00:05:00    315631
0 days 01:25:00         1
Name: count, dtype: int64


In [20]:
time_diff = df.index.to_series().diff()

gap = time_diff[time_diff != pd.Timedelta(minutes=5)]

print(gap)

Open Time
2022-12-31 18:30:00               NaT
2023-03-24 14:00:00   0 days 01:25:00
Name: Open Time, dtype: timedelta64[us]


In [21]:
gap_time = pd.Timestamp("2023-03-24 14:00:00")

location = df.index.get_loc(gap_time)

df.iloc[location-3:location+3]

,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Volume,Taker Buy Quote Volume
Open Time,,,,,,,,,,
2023-03-24 12:25:00,28080.00,28080.00,28080.00,28080.00,0.00000,2023-03-24 12:29:59.999,0.000000e+00,0,0.00000,0.000000e+00
2023-03-24 12:30:00,28080.00,28080.00,28080.00,28080.00,0.00000,2023-03-24 12:34:59.999,0.000000e+00,0,0.00000,0.000000e+00
2023-03-24 12:35:00,28080.00,28080.00,28080.00,28080.00,0.00000,2023-03-24 12:39:41.646,0.000000e+00,0,0.00000,0.000000e+00
2023-03-24 14:00:00,28079.99,28079.99,27835.00,27858.24,1209.62045,2023-03-24 14:04:59.999,3.374709e+07,24077,346.49241,9.669154e+06
2023-03-24 14:05:00,27858.23,27950.00,27857.58,27916.47,571.88072,2023-03-24 14:09:59.999,1.596052e+07,10546,306.90519,8.565686e+06
2023-03-24 14:10:00,27916.47,28253.01,27911.97,28160.01,1169.30835,2023-03-24 14:14:59.999,3.283858e+07,20664,737.35329,2.070980e+07


In [ ]:
df[df["Number of Trades"] == 0]

,Open,High,Low,Close,Volume,Close Time,Quote Asset Volume,Number of Trades,Taker Buy Base Volume,Taker Buy Quote Volume
Open Time,,,,,,,,,,
2023-03-24 11:30:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:34:59.999,0.0,0,0.0,0.0
2023-03-24 11:35:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:39:59.999,0.0,0,0.0,0.0
2023-03-24 11:40:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:44:59.999,0.0,0,0.0,0.0
2023-03-24 11:45:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:49:59.999,0.0,0,0.0,0.0
2023-03-24 11:50:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:54:59.999,0.0,0,0.0,0.0
2023-03-24 11:55:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 11:59:59.999,0.0,0,0.0,0.0
2023-03-24 12:00:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 12:04:59.999,0.0,0,0.0,0.0
2023-03-24 12:05:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 12:09:59.999,0.0,0,0.0,0.0
2023-03-24 12:10:00,28080.0,28080.0,28080.0,28080.0,0.0,2023-03-24 12:14:59.999,0.0,0,0.0,0.0


In [25]:
expected = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="5min"
)

missing = expected.difference(df.index)

print(missing)

DatetimeIndex(['2023-03-24 12:40:00', '2023-03-24 12:45:00',
               '2023-03-24 12:50:00', '2023-03-24 12:55:00',
               '2023-03-24 13:00:00', '2023-03-24 13:05:00',
               '2023-03-24 13:10:00', '2023-03-24 13:15:00',
               '2023-03-24 13:20:00', '2023-03-24 13:25:00',
               '2023-03-24 13:30:00', '2023-03-24 13:35:00',
               '2023-03-24 13:40:00', '2023-03-24 13:45:00',
               '2023-03-24 13:50:00', '2023-03-24 13:55:00'],
              dtype='datetime64[us]', freq='5min')


In [ ]:
exchange = ccxt.bybit()

ohlcv = exchange.fetch_ohlcv(
    'BTC/USDT',
    timeframe='5m',
    since=exchange.parse8601('2023-03-24T12:40:00Z'),
    limit=20
)

df = pd.DataFrame(
    ohlcv,
    columns=[
        "timestamp",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
)

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
print("ByBit Exchange Data: ")
print()
print(df)

ByBit Exchange Data: 

             timestamp      Open      High       Low     Close      Volume
0  2023-03-24 12:40:00  27940.97  28026.15  27914.01  28016.09   62.325252
1  2023-03-24 12:45:00  28016.09  28042.02  27974.65  28006.65   81.581113
2  2023-03-24 12:50:00  28006.65  28006.65  27958.02  27964.92   20.937888
3  2023-03-24 12:55:00  27964.92  28016.11  27964.91  27969.38   19.270613
4  2023-03-24 13:00:00  27969.38  27986.37  27914.53  27914.53   33.361064
5  2023-03-24 13:05:00  27914.53  27950.00  27871.31  27922.85   39.925953
6  2023-03-24 13:10:00  27922.85  27927.95  27883.23  27884.21   23.886663
7  2023-03-24 13:15:00  27884.21  27919.96  27737.60  27779.75   95.644659
8  2023-03-24 13:20:00  27779.75  27832.89  27706.13  27800.55   87.467372
9  2023-03-24 13:25:00  27800.55  27921.27  27783.95  27898.33   34.431304
10 2023-03-24 13:30:00  27898.33  27921.26  27845.89  27874.54   24.871217
11 2023-03-24 13:35:00  27874.54  27893.05  27786.13  27786.13   36.389894
12

In [2]:

exchange = ccxt.coinbase()

ohlcv = exchange.fetch_ohlcv(
    'BTC/USDT',
    timeframe='5m',
    since=exchange.parse8601('2023-03-24T12:40:00Z'),
    limit=20
)

df = pd.DataFrame(
    ohlcv,
    columns=[
        "timestamp",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
)

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
print("Coinbase Exchange Data: ")
print()
print(df)

Coinbase Exchange Data: 

             timestamp      Open      High       Low     Close     Volume
0  2023-03-24 12:40:00  27932.02  28019.93  27921.60  27997.09  10.958905
1  2023-03-24 12:45:00  28006.18  28038.27  27990.06  28007.41   7.048360
2  2023-03-24 12:50:00  27996.21  27996.39  27971.09  27971.09   1.861593
3  2023-03-24 12:55:00  27970.23  28010.75  27970.23  27972.49   4.759529
4  2023-03-24 13:00:00  27953.69  27984.93  27926.70  27926.70   2.454821
5  2023-03-24 13:05:00  27926.70  27956.45  27890.23  27918.46   2.717892
6  2023-03-24 13:10:00  27915.64  27915.64  27840.71  27840.71   2.959891
7  2023-03-24 13:15:00  27856.54  27926.57  27744.82  27765.76   4.004068
8  2023-03-24 13:20:00  27781.86  27839.16  27709.00  27795.23  10.060609
9  2023-03-24 13:25:00  27805.94  27934.70  27805.94  27904.64  10.629298
10 2023-03-24 13:30:00  27889.91  27919.37  27843.80  27875.36   8.028627
11 2023-03-24 13:35:00  27878.99  27898.73  27784.53  27789.68   5.201914
12 2023-03-2